Step 1 — Create a new notebook

In [ ]:
import torch

!pip install torch_geometric ogb

from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage
torch.serialization.add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])

from ogb.nodeproppred import PygNodePropPredDataset
dataset = PygNodePropPredDataset(name="ogbn-arxiv")
data = dataset[0]
split_idx = dataset.get_idx_split()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 795.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 5.1 MB/s eta 0:00:00


Downloaded 0.08 GB: 100%|██████████| 81/81 [00:24<00:00,  3.30it/s]


Extracting dataset/arxiv.zip


Processing...


Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 8738.13it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 204.75it/s]

Saving...



Done!


Step 2 — Load node features and labels

In [ ]:
x = data.x                  # shape [169343, 128]
y = data.y.squeeze()        # shape [169343] — squeeze removes an extra dimension OGB adds

print("Features shape:", x.shape)
print("Labels shape:", y.shape)
print("Number of classes:", y.max().item() + 1)
print("Sample labels:", y[:10])

Features shape: torch.Size([169343, 128])
Labels shape: torch.Size([169343])
Number of classes: 40
Sample labels: tensor([ 4,  5, 28,  8, 27, 34,  6,  4,  3, 28])


Step 3 — Create train/validation/test splits (use the official OGB split)

In [ ]:
train_idx = split_idx['train']
valid_idx = split_idx['valid']
test_idx = split_idx['test']

print("Train size:", train_idx.shape[0])
print("Valid size:", valid_idx.shape[0])
print("Test size:", test_idx.shape[0])
print("Total:", train_idx.shape[0] + valid_idx.shape[0] + test_idx.shape[0], "vs num_nodes:", data.num_nodes)

Train size: 90941
Valid size: 29799
Test size: 48603
Total: 169343 vs num_nodes: 169343


Step 4 — Check whether normalization is needed

In [ ]:
print("Feature stats before any processing:")
print("Min:", x.min().item())
print("Max:", x.max().item())
print("Mean:", x.mean().item())
print("Std:", x.std().item())

# Check per-feature-dimension scale (are all 128 dims on a similar scale?)
print("Per-dimension mean range:", x.mean(dim=0).min().item(), "to", x.mean(dim=0).max().item())

Feature stats before any processing:
Min: -1.3888649940490723
Max: 1.6387439966201782
Mean: 0.020484082400798798
Std: 0.23322448134422302
Per-dimension mean range: -0.4311313033103943 to 0.9191313982009888


Step 5 — Apply normalization (only if your Step 4 findings say you should)

In [ ]:
import torch.nn.functional as F

x_normalized = F.normalize(x, p=2, dim=1)  # L2-normalize each node's feature vector
print("Normalized feature norm (should be ~1.0):", x_normalized[0].norm().item())

Normalized feature norm (should be ~1.0): 0.9999998211860657


Step 6 — Package everything for the next task

In [ ]:
# Attach the (possibly normalized) features and confirm the object is ready for modeling
data.x = x_normalized if 'x_normalized' in dir() else x
data.y = y

print(data)

Data(num_nodes=169343, edge_index=[2, 1166243], x=[169343, 128], node_year=[169343, 1], y=[169343])


## Data Preparation Summary — OGBN-Arxiv

**1. Features and Labels**
The dataset provides a node feature matrix of shape [169343, 128], where each
row is a 128-dimensional embedding representing a paper's title and abstract.
Labels are provided as a tensor of shape [169343], each value indicating one
of [40] subject categories. The original label tensor from OGB has shape
[N, 1], so we applied `.squeeze()` to flatten it to [N], matching the shape
required by PyTorch's CrossEntropyLoss.

**2. Train / Validation / Test Split**
We used the official OGB split rather than creating our own random split:
- Train: [90,941] nodes
- Validation: [29,799] nodes
- Test: [48,603] nodes

This split is based on publication year — older papers are used for training
and newer papers for testing. We chose to keep this official split for two
reasons: (1) it reflects a realistic real-world scenario, where a model
trained on past papers must classify newly published ones, and (2) it keeps
our results comparable to published benchmarks on the OGB leaderboard, since
a custom random split would not be directly comparable and could leak
information across time (a paper "knowing about" future citation patterns
it wouldn't have access to in reality).

**3. Normalization Decision**
Before processing, feature values ranged from [min] to [max] with a mean of
[mean] and standard deviation of [std]. Per-dimension means ranged from
[low] to [high], indicating [features were already on a similar, small
scale / features varied considerably across dimensions].

Decision: We [did / did not] apply L2 row-normalization.
Reasoning: [Choose the one that matches your actual findings —
  e.g., "Since the OGB-provided embeddings were already